In [2]:
import os
import math
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn.functional as F  # standard alias
# from a pythonfile import a class or function
from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADVisualizer   import CADVisualizer
from HDVClassNet.PP_net import PPNet
from HDVClassNet.VoronoiDecorder import VoronoiDecoder

from Training.MainTrain import TrainingConfig, NN_Trainer
from neuraltomo_fem import run_fem_loss
from problems.TipCantilever_30_20_20_midLoad import TipCantilever_30_20_20_midLoad
from problems.ThickenShell import ThickenShell

import pyvista as pv
try:
    pv.set_jupyter_backend("trame")
except Exception:
    pass

# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)
# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


Code Directory: /home/arash/VoronoiImp-main
Test Step files Directory: /home/arash/VoronoiImp-main/Testparts
device: cuda


In [3]:
viz = CADVisualizer()
# Laoding model and extracting mesh and tensors as input
FreeFormSurf1  = TesPartsDir / "FreeFormCrv1.stp"
FreeFormSurf2A = TesPartsDir / "FreeFormSurf2A.STEP"
YachtBodypart  = TesPartsDir / "YachtBodypart.stp"
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Cube           = TesPartsDir / "Cube.stp"
CircularSur2   = TesPartsDir / "CircularSur2.stp"
Conic          = TesPartsDir / "Conic.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
FullCylinder   = TesPartsDir / "FullCylinder.stp"
Sphere         = TesPartsDir / "Sphere.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Tidebottle     = TesPartsDir / "Tidebottle.STEP"

shape_path = str(SphereTap)
generator = CADTensorGenerator(
    deflection=0.1,
    angle=0.5,
    metric_tol=1e-9,
    det_min=1e-5,
    n_u=70,
    n_v=70,
    device=device,
)
mesh_df, faces_df, tensors = generator.generate_from_file(shape_path, input_ring=1)

# mesh_df, faces_df, tensors = generator.generate_from_file(
#     shape_path=shape_path,
#     input_ring=1,
#     visualize=True,
# )
print(tensors["num_faces"])
print(tensors["face_tensors"][0].keys())
print(tensors["face_periodicity"])
uv = tensors["uv"]
points_xyz = tensors["points_xyz"]
uv = tensors["uv"]
Xu = tensors["Xu"]
Xv = tensors["Xv"]
points_xyz = tensors["points_xyz"]
face_areas = tensors["face_areas"]
faces_ijk = tensors["faces_ijk"]
face_id = tensors["face_id"]
boundary_idx_ring1 = tensors["boundary_idx_ring1"]
pv_faces = tensors["pv_faces"]
viz.visualize_show_Model(points_xyz, pv_faces)
tensors["BBX"]
print("BBX:", tensors["BBX"])

MinVolFrac: 0.13605931401252747
uv device: cuda:0
1
dict_keys(['face_id', 'uv', 'points_xyz', 'Xu', 'Xv', 'faces_ijk', 'pv_faces', 'face_areas', 'boundary_idx', 'boundary_idx_ring1', 'boundary_idx_ring2', 'min_vol_frac', 'BBX', 'u_periodic', 'v_periodic', 'u_period', 'v_period', 'u_raw_bounds', 'v_raw_bounds', 'global_vertex_idx', 'global_face_idx', 'num_vertices', 'num_faces'])
{0: {'u_periodic': True, 'v_periodic': False, 'u_period': 6.28318530717959, 'v_period': None}}


Widget(value='<iframe src="http://localhost:38815/index.html?ui=P_0x7e0e400b6e90_0&reconnect=auto" class="pyvi…

BBX: {0: {'xmin': -27.0, 'xmax': 26.999999999999975, 'ymin': -27.0, 'ymax': 26.999999999999996, 'zmin': -20.0, 'zmax': 20.0}}


In [ ]:
fixed_height_shell= 1
shell_problem = ThickenShell(
    thickness=fixed_height_shell,
    BC_dir = "z",
    Load_magnitude=1.0,
    voxel_size=1.0,
    extra_layers=2,
    tensors=tensors,
    tangential_tol=1,
)

fem = run_fem_loss.NeuralTOMOFEM(shell_problem, device=device, isotropic=False)
shell_problem.debug_voxel_stats()
shell_problem.show_voxels_surface_and_bc_NEW()

Widget(value='<iframe src="http://localhost:38815/index.html?ui=P_0x7e0e3e34f6a0_3&reconnect=auto" class="pyvi…

In [8]:
cfg = TrainingConfig(
    seed_number=10,
    use_anisotropy=False,
    fixed_height=fixed_height_shell,
    target_volfrac=0.3,
    seed_repulsion_sigma=0.08,
    boundary_margin=0.05,
    w_min= 0.005,
    w_max= 0.5,

    lam_fem=10.0,
    lam_vol=1.0,
    lam_rep=0.2,
    lam_bnd=0.01,

    # NEW: strutness loss
    lam_strut=0.02,
    lam_strut_edge=1.0,
    lam_strut_void=0.2,
    strut_edge_power=1.0,
    strut_void_power=1.0,
    strut_void_from_one_minus_edge=True,

        
    lr_seed_refine = 2e-3,
    lr_delta_head = 2e-4,
    lr_mlp = 2e-4,
    lr_w_head = 2e-4,
    lr_h_head = 2e-4,

    comp_normalize_by=1e10,
    normalize_losses=False,

    fem_density_floor=0.02,
    skip_bad_fem_steps=True,

    num_steps=1000,
    context_vector_size=8,

    # tau only controls soft Voronoi ownership now
    tau=0.1,

    # beta now controls sharpness of the geometric bisector band
    beta=0.02,

    log_every=50,
    early_stop_start=500,
    patience=500,
    min_delta=1e-8,
)
trainer = NN_Trainer(
    generator=generator,
    viz=viz,
    decoder_cls=VoronoiDecoder,
    ppnet_cls=PPNet,
    fem=fem,
    shell_problem=shell_problem,
    config=cfg,
)

result = trainer.train(
    uv=uv,
    Xu=Xu,
    Xv=Xv,
    points_xyz=points_xyz,
    face_areas=face_areas,
    faces_ijk=faces_ijk,
    face_id=face_id,
    boundary_idx_ring1=boundary_idx_ring1,
    face_tensors=tensors["face_tensors"],
)

trainer.visualize_result_stepwise(result, points_xyz, faces_ijk)
trainer.visualize_result_final(result, points_xyz, faces_ijk, thr=0.5, show_solid=False)

[00000] L_total=1.0717e-02 | L_vol=2.011e-03 L_fem=1.450e-04 L_strut=2.388e-01  L_rep=1.145e-05 L_bnd=2.478e-01 |vol=0.345 (/0.300) comp=1.450e+06 w=2.537e-01 | Lse=1.887e-01 Lsv=2.504e-01 rho(min/mean/max)=0.001/0.391/0.827 | Δrho=0.00e+00 Δseed=0.00e+00 grad_mean=2.08e-04 | fem=OK | best=1.0717e-02@0 
[00050] L_total=8.9249e-03 | L_vol=1.156e-03 L_fem=1.368e-04 L_strut=2.377e-01  L_rep=1.278e-05 L_bnd=1.645e-01 |vol=0.334 (/0.300) comp=1.368e+06 w=2.515e-01 | Lse=1.897e-01 Lsv=2.397e-01 rho(min/mean/max)=0.001/0.380/0.828 | Δrho=9.79e-02 Δseed=2.70e-02 grad_mean=3.43e-04 | fem=OK | best=8.9111e-03@46 
[step 85] L_total is non-finite, optimizer step skipped.

=== FEM FAILURE AT STEP 85 ===
rho_surface_shape: (1185,)
fiber_surface_shape: (1185, 3)
density_shape: (156645,)
phi_shape: (156645,)
theta_shape: (156645,)
density_floor: 0.02
density_raw_min: nan
density_raw_mean: nan
density_raw_max: nan
density_min: nan
density_mean: nan
density_max: nan
phi_has_nan: True
phi_has_inf: False


Widget(value='<iframe src="http://localhost:38815/index.html?ui=P_0x7e0aa585ddb0_4&reconnect=auto" class="pyvi…

threshold=0.5 (manual) | solid%=46.498%


Widget(value='<iframe src="http://localhost:38815/index.html?ui=P_0x7e0bf42dc730_5&reconnect=auto" class="pyvi…

(UnstructuredGrid (0x7e0bc6b54940)
   N Cells:    1366
   N Points:   909
   X Bounds:   -2.700e+01, 2.700e+01
   Y Bounds:   -2.700e+01, 2.700e+01
   Z Bounds:   -2.000e+01, 2.000e+01
   N Arrays:   3,
 0.5)

In [5]:
cfg = TrainingConfig(
    seed_number=15,
    use_anisotropy=False,
    fixed_height=fixed_height_shell,
    target_volfrac=0.08,
    seed_repulsion_sigma=0.08,
    boundary_margin=0.05,
    w_min= 0.005,
    w_max= 0.5,

    lam_fem=10.0,
    lam_vol=1.0,
    lam_rep=0.2,
    lam_bnd=0.01,

    # NEW: strutness loss
    lam_strut=0.02,
    lam_strut_edge=1.0,
    lam_strut_void=0.2,
    strut_edge_power=1.0,
    strut_void_power=1.0,
    strut_void_from_one_minus_edge=True,

        
    lr_seed_refine = 2e-3,
    lr_delta_head = 2e-4,
    lr_mlp = 2e-4,
    lr_w_head = 2e-4,
    lr_h_head = 2e-4,

    comp_normalize_by=1e10,
    normalize_losses=False,

    fem_density_floor=0.02,
    skip_bad_fem_steps=True,

    num_steps=1000,
    context_vector_size=8,

    # tau only controls soft Voronoi ownership now
    tau=0.1,

    # beta now controls sharpness of the geometric bisector band
    beta=0.02,

    log_every=50,
    early_stop_start=500,
    patience=500,
    min_delta=1e-8,
)
trainer = NN_Trainer(
    generator=generator,
    viz=viz,
    decoder_cls=VoronoiDecoder,
    ppnet_cls=PPNet,
    fem=fem,
    shell_problem=shell_problem,
    config=cfg,
)

result = trainer.train(
    uv=uv,
    Xu=Xu,
    Xv=Xv,
    points_xyz=points_xyz,
    face_areas=face_areas,
    faces_ijk=faces_ijk,
    face_id=face_id,
    boundary_idx_ring1=boundary_idx_ring1,
    face_tensors=tensors["face_tensors"],
)

trainer.visualize_result_stepwise(result, points_xyz, faces_ijk)
trainer.visualize_result_final(result, points_xyz, faces_ijk, thr=0.5, show_solid=False)

[00000] L_total=6.0085e-02 | L_vol=4.906e-02 L_fem=3.977e-05 L_strut=2.465e-01  L_rep=1.223e-05 L_bnd=5.694e-01 |vol=0.301 (/0.080) comp=3.977e+05 w=2.523e-01 | Lse=2.132e-01 Lsv=1.665e-01 rho(min/mean/max)=0.000/0.308/0.874 | Δrho=0.00e+00 Δseed=0.00e+00 grad_mean=4.10e-04 | fem=OK | best=6.0085e-02@0 
[00050] L_total=5.2634e-02 | L_vol=4.134e-02 L_fem=1.431e-05 L_strut=2.496e-01  L_rep=1.576e-03 L_bnd=5.843e-01 |vol=0.283 (/0.080) comp=1.431e+05 w=2.481e-01 | Lse=2.186e-01 Lsv=1.551e-01 rho(min/mean/max)=0.000/0.290/0.864 | Δrho=2.47e-01 Δseed=8.42e-02 grad_mean=5.68e-04 | fem=OK | best=5.1980e-02@25 
[00100] L_total=5.2413e-02 | L_vol=4.126e-02 L_fem=1.783e-05 L_strut=2.453e-01  L_rep=1.979e-03 L_bnd=5.671e-01 |vol=0.283 (/0.080) comp=1.783e+05 w=2.444e-01 | Lse=2.141e-01 Lsv=1.561e-01 rho(min/mean/max)=0.000/0.290/0.860 | Δrho=2.80e-01 Δseed=8.28e-02 grad_mean=1.09e-04 | fem=OK | best=5.1980e-02@25 
[00150] L_total=5.2003e-02 | L_vol=4.072e-02 L_fem=1.508e-05 L_strut=2.486e-01  L_r

Widget(value='<iframe src="http://localhost:42803/index.html?ui=P_0x7ebce86af940_4&reconnect=auto" class="pyvi…

threshold=0.5 (manual) | solid%=19.306%


Widget(value='<iframe src="http://localhost:42803/index.html?ui=P_0x7ebef6ad3c40_5&reconnect=auto" class="pyvi…

(UnstructuredGrid (0x7ebc7a44ec80)
   N Cells:    10218
   N Points:   6393
   X Bounds:   0.000e+00, 1.000e+01
   Y Bounds:   -1.275e-01, 2.101e+00
   Z Bounds:   -2.033e+00, 1.939e+00
   N Arrays:   3,
 0.5)